*0.3 Classical NLP*

# Tokenization: BPE

**The situation.** You are fine-tuning a model for a pharma client. Their documents are full of drug names that the base tokenizer shreds into 6–8 pieces each, so every prompt costs three times more and the model handles the names poorly. A tokenizer trained on their text would learn those names as single pieces.

**Byte-pair encoding (BPE).** Start with single characters. Count every pair of neighbouring pieces in the training text; merge the most frequent pair into a new piece; repeat until the vocabulary is the size you want. Frequent words and word-parts become single pieces; rare ones stay split. GPT models use BPE.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Train a BPE tokenizer on a small corpus** with the `tokenizers` library (the same code that built GPT-2's). Watch which merges it learns first.

In [2]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

corpus = [
    "The customer was charged twice for the order.",
    "Please refund the duplicate charge on the invoice.",
    "Charged twice, refund requested, invoice attached.",
    "The refund for the duplicate invoice was processed.",
    "Chargebacks and bank refunds are checked weekly.",
] * 20  # repeated so the counts are clear

tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = (
    pre_tokenizers.Whitespace()
)  # split on spaces first; merges happen inside words
trainer = trainers.BpeTrainer(vocab_size=80, special_tokens=["[UNK]"])
tokenizer.train_from_iterator(corpus, trainer)

vocabulary = tokenizer.get_vocab()
learned = []
for piece, _ in sorted(vocabulary.items(), key=lambda item: item[1]):
    if len(piece) > 1 and piece != "[UNK]":
        learned.append(piece)
print("vocabulary size:", len(vocabulary))
print("first merges learned:", learned[:12])
print("'refunded' →", tokenizer.encode("refunded").tokens)
print("'chargeback' →", tokenizer.encode("chargeback").tokens)
assert "refund" in vocabulary


vocabulary size: 80
first merges learned: ['he', 'ic', 're', 'ed', 'nd', 'ice', 'ar', 'fu', 'har', 'the', 'refu', 'harg']
'refunded' → ['refund', 'ed']
'chargeback' → ['charg', 'ebac', 'k']




**Reading the output.** The earliest merges are the most frequent pairs in this corpus — pieces of "the", "charge", "refund", "invoice". "refund" became a single piece because it appeared often; "refunded" is that piece plus a leftover; "chargeback" (never seen) still tokenizes, from the pieces it does know.

```
characters   r e f u n d
merge 1      re f u n d        ("r","e" was the commonest pair)
merge 2      re fu n d
   …
final        refund            one piece
```

**The rule to remember.** BPE builds the vocabulary from what is frequent in the training text. Train it on the domain and the domain's words become cheap single pieces.

| Use it when | Don't when | Instead use |
|---|---|---|
| training a model from scratch on a domain; understanding how GPT tokenizers behave | using an existing model — its tokenizer is fixed | the model's own tokenizer |

**Watch out**
- A new tokenizer means a new model; you cannot swap the tokenizer under a trained model.
- `vocab_size` is a trade-off: bigger → fewer tokens per text but a larger embedding table and rarer pieces trained less.
- Byte-level BPE (GPT-2 onward) starts from bytes, not characters, so any input — emoji, any script — is representable.